# Brain Tumor MRI Classification
ResNet50 Transfer Learning | 4 Classes: Glioma, Meningioma, Pituitary, No Tumor

In [ ]:
import tensorflow as tf
print('TF Version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Dataset

In [ ]:
import os

# Auto-detect Kaggle vs local environment
DATASET_PATH = None
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'Training' in dirs:
            DATASET_PATH = root
            break
    print('Running on Kaggle')
else:
    import kagglehub
    DATASET_PATH = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
    print('Running locally')

print('Dataset path:', DATASET_PATH)
print('Contents:', os.listdir(DATASET_PATH))

In [ ]:
train_path = os.path.join(DATASET_PATH, 'Training')
test_path  = os.path.join(DATASET_PATH, 'Testing')

print('Training classes:')
for cls in sorted(os.listdir(train_path)):
    count = len(os.listdir(os.path.join(train_path, cls)))
    print(f'  {cls}: {count} images')

print('\nTesting classes:')
for cls in sorted(os.listdir(test_path)):
    count = len(os.listdir(os.path.join(test_path, cls)))
    print(f'  {cls}: {count} images')

## 2. Visualize Samples

In [ ]:
import cv2
import random
import matplotlib.pyplot as plt

classes = sorted(os.listdir(train_path))
fig, axes = plt.subplots(1, len(classes), figsize=(16, 4))

for i, cls in enumerate(classes):
    img_name = random.choice(os.listdir(os.path.join(train_path, cls)))
    img = cv2.imread(os.path.join(train_path, cls, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=12)
    axes[i].axis('off')

plt.suptitle('Sample MRI Images per Class', fontsize=14)
plt.tight_layout()
plt.savefig('sample_images.png', bbox_inches='tight')
plt.show()

## 3. Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE   = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]  # helps with MRI brightness variation
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Class indices:', train_generator.class_indices)

## 4. Build Model

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, BatchNormalization
from tensorflow.keras.models import Model

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False  # Freeze for Phase 1

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
print(f'Total layers: {len(model.layers)}')
print(f'Trainable layers (Phase 1): {sum(1 for l in model.layers if l.trainable)}')

## 5. Phase 1 — Train Head Only (base frozen)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_p1 = ModelCheckpoint(
    'best_model_phase1.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

history_p1 = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=10,
    callbacks=[checkpoint_p1]
)

print(f'\nPhase 1 best val_accuracy: {max(history_p1.history["val_accuracy"]):.4f}')

## 6. Phase 2 — Fine-tune Top Layers (unfreeze last 30)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f'Trainable layers (Phase 2): {sum(1 for l in model.layers if l.trainable)}')

model.compile(
    optimizer=Adam(learning_rate=0.00001),  # much lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_p2 = ModelCheckpoint(
    'best_model_final.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-7,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

history_p2 = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=30,
    callbacks=[checkpoint_p2, reduce_lr, early_stop]
)

print(f'\nPhase 2 best val_accuracy: {max(history_p2.history["val_accuracy"]):.4f}')

## 7. Training Curves

In [ ]:
import numpy as np

acc      = history_p1.history['accuracy']     + history_p2.history['accuracy']
val_acc  = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
loss     = history_p1.history['loss']         + history_p2.history['loss']
val_loss = history_p1.history['val_loss']     + history_p2.history['val_loss']
phase2_start = len(history_p1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, train, val, title in [
    (ax1, acc, val_acc, 'Accuracy'),
    (ax2, loss, val_loss, 'Loss')
]:
    ax.plot(train, label='Train')
    ax.plot(val,   label='Validation')
    ax.axvline(x=phase2_start, color='red', linestyle='--', label='Phase 2 start')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 8. Evaluate & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

loss, acc = model.evaluate(test_generator, verbose=1)
print(f'\nFinal Test Accuracy: {acc:.4f} ({acc*100:.2f}%)')

y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print('\nClassification Report:')
print(classification_report(y_true, y_pred_classes, target_names=class_names))

cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 9. Save Final Model

In [ ]:
model.save('brain_tumor_resnet50_final.keras')
print('Model saved: brain_tumor_resnet50_final.keras')
print(f'Best val accuracy: {max(history_p2.history["val_accuracy"]):.4f}')